In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))


from utils.config import Configuration, load_config

In [ ]:
config = load_config("../config.yaml")

In [ ]:
import pandas as pd

for subject_i in range(1, 8+1):
    faces_path = os.path.join("..",config.directories.excel_files_target_dir, f"subj_{subject_i:02d}", config.dataset_creation.subset_animate_face_final)

    non_faces_path = os.path.join("..",config.directories.excel_files_target_dir, f"subj_{subject_i:02d}", config.dataset_creation.subset_animate_non_face_final)

    faces_df = pd.read_excel(faces_path)
    non_faces_df = pd.read_excel(non_faces_path)

    print(f"Subject {subject_i} has {len(faces_df)} animate face images and {len(non_faces_df)} animate non-face images")



In [ ]:
import numpy as np
import nibabel as nib
import os
import time
import logging
from t_testing.clean_roi_mask import modify_mask_with_ttest


def create_rdm(
    config: Configuration,
    list_subj,
    set_to_take: str,
    t_test_threshold: float,
    mode="averaged",
    sample_to_pick: int = 0,
    randomization: bool = False,
):
    assert mode in ["averaged", "single"]
    logging.info(f"Creating RDM for mode {mode} - T-Test THRESHOLD: {t_test_threshold}")
    logging.info(f"Using distance-metric={config.pipeline.step_4_rsa_analysis.distance_metric}")
    time.sleep(0.25)
    logging.info(f"Using positive set: {os.path.join(set_to_take, config.dataset_creation.subset_animate_face_final)}")

    if mode == "single":
        logging.info(f"Picking sample {sample_to_pick}")

    for i, sub in enumerate(list_subj):
        # Determine if using shared set or not
        pick_shared = set_to_take == "shared"

        # Modify and generate temporary t-test mask files
        modify_mask_with_ttest(
            config, t_test_threshold, pick_shared, sub, sub_filename="temporary_mask"
        )

        # Load left and right hemisphere masks
        mask_path_lh = os.path.join(
            config.directories.t_test_roi_dir,
            set_to_take,
            f"lh.subj{sub:02d}.temporary_mask.mgz",
        )
        mask_path_rh = os.path.join(
            config.directories.t_test_roi_dir,
            set_to_take,
            f"rh.subj{sub:02d}.temporary_mask.mgz",
        )
        logging.info(f"Loading mask from\n{mask_path_rh=}\n{mask_path_lh=}")

        mask_lh = nib.load(mask_path_lh).get_fdata().squeeze()
        mask_rh = nib.load(mask_path_rh).get_fdata().squeeze()

        # Combine and compute unique values
        combined_mask = np.concatenate([mask_lh.flatten(), mask_rh.flatten()])
        unique_vals = np.unique(combined_mask)
        logging.info(f"Subject {sub:02d} - Unique values in combined mask: {unique_vals}")


for subject_i in range(1, 8+1):
    create_rdm(config, [subject_i], f"subj_{subject_i:02d}", 3, mode="averaged")